In [1]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/datasets/ngocccchannn/lightgcn-backbone-warm/g_i_warm.npy
/kaggle/input/datasets/ngocccchannn/lightgcn-backbone-warm/g_u_warm.npy
/kaggle/input/datasets/ngocccchannn/lightgcn-backbone-warm/user_index_warm.json
/kaggle/input/datasets/ngocccchannn/warm-scenario/test_warm.parquet
/kaggle/input/datasets/ngocccchannn/warm-scenario/train_warm.parquet
/kaggle/input/datasets/ngocccchannn/warm-scenario/val_warm.parquet
/kaggle/input/datasets/ngocccchannn/lightgcn-backbone-cold/g_i.npy
/kaggle/input/datasets/ngocccchannn/lightgcn-backbone-cold/g_u.npy
/kaggle/input/datasets/ngocccchannn/lightgcn-backbone-cold/item_index.json
/kaggle/input/datasets/ngocccchannn/lightgcn-backbone-cold/user_index_cold.json
/kaggle/input/datasets/ngocccchannn/evaluator/evaluator.py


In [2]:
import os
import sys
import json
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
import random

def set_seed(s=42):
    random.seed(s)
    np.random.seed(s)
    torch.manual_seed(s)
    torch.cuda.manual_seed_all(s)
set_seed(42)
print("Đã cố định random seed!")

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Sử dụng thiết bị: {DEVICE}")

sys.path.append('/kaggle/input/datasets/ngocccchannn/evaluator') 
from evaluator import evaluate

Đã cố định random seed!
Sử dụng thiết bị: cuda


In [3]:
# Load dữ liệu warm
tr_df = pd.read_parquet('/kaggle/input/datasets/ngocccchannn/warm-scenario/train_warm.parquet')
va_df = pd.read_parquet('/kaggle/input/datasets/ngocccchannn/warm-scenario/val_warm.parquet')
te_df = pd.read_parquet('/kaggle/input/datasets/ngocccchannn/warm-scenario/test_warm.parquet')

# Xây dựng ánh xạ user2id và item2id CHỈ dựa trên tập Train
with open('/kaggle/input/datasets/ngocccchannn/lightgcn-backbone-warm/user_index_warm.json', 'r') as f:
    warm_user_list = json.load(f)

with open('/kaggle/input/datasets/ngocccchannn/lightgcn-backbone-cold/item_index.json', 'r') as f:
    global_item_list = json.load(f)

# Lọc danh sách item từ index chung chỉ giữ lại các khóa có xuất hiện trong train_warm (2.448 khóa)
train_course_set = set(tr_df['course_id'].astype(str))
filtered_item_list = [item for item in global_item_list if str(item) in train_course_set]

warm_item2id = {str(item): i for i, item in enumerate(filtered_item_list)}
warm_user2id = {str(u): i for i, u in enumerate(warm_user_list)}


num_users = len(warm_user2id)
num_items = len(warm_item2id)
print(f"Tổng số Users (Train): {num_users}, Tổng số Items (Train): {num_items}")

# Chuẩn bị tensor tương tác cho tập train warm
train_users = torch.tensor([warm_user2id[str(u)] for u in tr_df['user_id'] if str(u) in warm_user2id], dtype=torch.long)
train_items = torch.tensor([warm_item2id[str(i)] for i in tr_df['course_id'] if str(i) in warm_item2id], dtype=torch.long)

# Tập khóa có trong train (warm_idx dùng cho chế độ uniform)
warm_idx = torch.unique(train_items)    # 2.448 khóa duy nhất
assert len(warm_idx) == num_items, f"Số lượng khóa duy nhất không khớp: warm_idx - {len(warm_idx)} vs num_items - {num_items}"


# Lịch sử tương tác (chỉ lọc các user/item có trong tập Train)
hist_warm_df = pd.concat([tr_df, va_df])
hist_warm = {}
for u, group in hist_warm_df.groupby('user_id'):
    if str(u) in warm_user2id:
        u_idx = warm_user2id[str(u)]
        item_idxs = [warm_item2id[str(i)] for i in group['course_id'] if str(i) in warm_item2id]
        if len(item_idxs) > 0:
            hist_warm[u_idx] = item_idxs

target_warm = {}
for u, group in te_df.groupby('user_id'):
    if str(u) in warm_user2id:
        u_idx = warm_user2id[str(u)]
        item_idxs = [warm_item2id[str(i)] for i in group['course_id'] if str(i) in warm_item2id]
        if len(item_idxs) > 0:
            target_warm[u_idx] = item_idxs

Tổng số Users (Train): 88729, Tổng số Items (Train): 2448


In [4]:
def shuffled_items(train_items, warm_idx, mode, seed):
    g = torch.Generator().manual_seed(seed)
    if mode == "permute":  # Giữ bậc học viên và bậc khóa -> mô hình chỉ học được độ phổ biến
        return train_items[torch.randperm(len(train_items), generator=g)]
    if mode == "uniform":  # Giữ bậc học viên, phá độ phổ biến
        return warm_idx[torch.randint(len(warm_idx), (len(train_items),), generator=g)]

In [5]:
class LightGCN(nn.Module):
    def __init__(self, num_users, num_items, latent_dim=64, n_layers=3):
        super(LightGCN, self).__init__()
        self.num_users = num_users
        self.num_items = num_items
        self.latent_dim = latent_dim
        self.n_layers = n_layers
        
        self.user_embedding = nn.Embedding(num_users, latent_dim)
        self.item_embedding = nn.Embedding(num_items, latent_dim)
        nn.init.xavier_uniform_(self.user_embedding.weight)
        nn.init.xavier_uniform_(self.item_embedding.weight)

    def forward(self, adj_matrix): 
        users_emb = self.user_embedding.weight
        items_emb = self.item_embedding.weight
        all_emb = torch.cat([users_emb, items_emb], dim=0)
        
        embs = [all_emb]
        for _ in range(self.n_layers):
            all_emb = torch.sparse.mm(adj_matrix, all_emb)
            embs.append(all_emb)
            
        lightgcn_emb = torch.stack(embs, dim=1).mean(dim=1)
        users_emb_final, items_emb_final = torch.split(lightgcn_emb, [self.num_users, self.num_items])
        return users_emb_final, items_emb_final

In [6]:
def bpr_loss(users, pos_items, neg_items, u_emb, i_emb):
    u_g = u_emb[users]
    pos_g = i_emb[pos_items]
    neg_g = i_emb[neg_items]
    pos_scores = torch.sum(u_g * pos_g, dim=1)
    neg_scores = torch.sum(u_g * neg_g, dim=1)
    return -torch.mean(torch.nn.functional.logsigmoid(pos_scores - neg_scores))

def expected_random_recall_support(target, mask, support, k=20):
    vals = []
    n_sup = int(support.sum())
    for u, gt in target.items():
        n = n_sup - sum(1 for i in mask.get(u, ()) if support[i])
        frac_in = sum(1 for i in gt if support[i]) / len(gt)
        vals.append(min(1.0, k / max(1, n)) * frac_in)
    return float(np.mean(vals))

In [7]:
# Thiết lập mảng support và ngưỡng kiểm tra theo Phụ lục A.2
train_item_indices = train_items.numpy()
support_train = np.zeros(num_items, dtype=bool)
support_train[train_item_indices] = True

ref = expected_random_recall_support(target_warm, hist_warm, support_train, k=20) # ~ 0.0083
threshold_uniform = 2 * ref # 0.0165
threshold_permute = 1.1 * 0.0753 # 0.0828 (với MostPop Recall@20 warm = 0.0753)

In [8]:
# Kiểm tra trước khi huấn luyện với seed 42: độ phổ biến sau khi xáo phải gần đều
cnt = torch.bincount(shuffled_items(train_items, warm_idx, "uniform", 42),
                     minlength=num_items).float()
cv = (cnt.std() / cnt.mean()).item()
print(f"Hệ số biến thiên sau khi xáo: {cv:.3f}")    
assert cv < 0.2, "Cách xáo uniform vẫn giữ độ phổ biến"

Hệ số biến thiên sau khi xáo: 0.094


In [9]:
SEEDS = [42, 2023, 2024, 1007, 999]
epochs = 60 
score_fn_dict = {"permute": {}, "uniform": {}}

print("\n--- Bắt đầu kiểm tra rò rỉ sử dụng hàm shuffled_items ---")
for mode in ("permute", "uniform"):
    print(f"\nĐang chạy chế độ xáo: [{mode}]")
    for s in SEEDS:
        # Gọi hàm shuffled_items 
        shuffled_train_items = shuffled_items(train_items, warm_idx, mode, s)
        shuffled_train_users = train_users

        row = torch.cat([shuffled_train_users, shuffled_train_items + num_users])
        col = torch.cat([shuffled_train_items + num_users, shuffled_train_users])
        val = torch.ones(row.size(0), dtype=torch.float32)

        assoc = torch.sparse_coo_tensor(torch.stack([row, col]), val, (num_users + num_items, num_users + num_items)).to(DEVICE)
        deg = torch.sparse.sum(assoc, dim=1).to_dense()
        deg_inv_sqrt = torch.pow(deg, -0.5)
        deg_inv_sqrt[deg_inv_sqrt == float('inf')] = 0.0

        i_idx = torch.arange(num_users + num_items, device=DEVICE)
        diag_indices = torch.stack([i_idx, i_idx], dim=0)
        deg_mat_inv_sqrt = torch.sparse_coo_tensor(diag_indices, deg_inv_sqrt.to(DEVICE), (num_users + num_items, num_users + num_items))
        adj_mat_s = torch.sparse.mm(deg_mat_inv_sqrt, torch.sparse.mm(assoc, deg_mat_inv_sqrt))

        model_shuffled = LightGCN(num_users, num_items, latent_dim=64, n_layers=3).to(DEVICE)
        optimizer_s = optim.Adam(model_shuffled.parameters(), lr=1e-3)

        model_shuffled.train()
        for epoch in range(epochs):
            perm_b = torch.randperm(len(shuffled_train_users))
            b_u = shuffled_train_users[perm_b].to(DEVICE)
            b_pos = shuffled_train_items[perm_b].to(DEVICE)
            b_neg = torch.randint(0, num_items, (len(perm_b),), dtype=torch.long).to(DEVICE)

            for batch_start in range(0, len(perm_b), 1024):
                end_idx = batch_start + 1024
                optimizer_s.zero_grad()
                u_emb_s, i_emb_s = model_shuffled(adj_mat_s)
                loss_s = bpr_loss(b_u[batch_start:end_idx], b_pos[batch_start:end_idx], b_neg[batch_start:end_idx], u_emb_s, i_emb_s)
                loss_s.backward()
                optimizer_s.step()

        model_shuffled.eval()
        with torch.no_grad():
            u_e, i_e = model_shuffled(adj_mat_s)
            def score_fn_s(users_tensor, u_e=u_e, i_e=i_e):
                u_idx = users_tensor.to(DEVICE)
                return torch.matmul(u_e[u_idx], i_e.t())
            score_fn_dict[mode][s] = score_fn_s


--- Bắt đầu kiểm tra rò rỉ sử dụng hàm shuffled_items ---

Đang chạy chế độ xáo: [permute]


/tmp/ipykernel_23/4026895519.py:17: UserWarning: Sparse invariant checks are implicitly disabled. Memory errors (e.g. SEGFAULT) will occur when operating on a sparse tensor which violates the invariants, but checks incur performance overhead. To silence this warning, explicitly opt in or out. See `torch.sparse.check_sparse_tensor_invariants.__doc__` for guidance.  (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:760.)
  assoc = torch.sparse_coo_tensor(torch.stack([row, col]), val, (num_users + num_items, num_users + num_items)).to(DEVICE)



Đang chạy chế độ xáo: [uniform]


In [10]:
cnt = torch.bincount(shuffled_items(train_items, warm_idx, "uniform", 42),
                     minlength=num_items).float()
cv = (cnt.std() / cnt.mean()).item()
print(f"Hệ số biến thiên sau khi xáo: {cv:.3f}")      # kỳ vọng ≈ 0,09 (dữ liệu gốc: 2,23)
assert cv < 0.2, "Cách xáo uniform vẫn giữ độ phổ biến"

Hệ số biến thiên sau khi xáo: 0.094


In [11]:
# Tính toán Recall@20 trung bình cho từng cách xáo 
r20_results = {}
for m in ("permute", "uniform"):
    r20_results[m] = np.mean([
        evaluate(score_fn_dict[m][s], target_warm, hist_warm, ks=(20,), tie_seed=s)["Recall@20"].mean()
        for s in SEEDS
    ])

ok_perm = r20_results["permute"] <= threshold_permute
ok_unif = r20_results["uniform"] <= threshold_uniform

print("\n================ KẾT QUẢ KIỂM TRA RÒ RỈ ================")
print(f"Cách 1 (Permute - Giữ độ phổ biến): Recall@20 = {r20_results['permute']:.4f} | Ngưỡng = {threshold_permute:.4f} --> {'ĐẠT' if ok_perm else 'KHÔNG ĐẠT'}")
print(f"Cách 2 (Uniform - Phá độ phổ biến): Recall@20 = {r20_results['uniform']:.4f} | Ngưỡng = {threshold_uniform:.4f} --> {'ĐẠT' if ok_unif else 'KHÔNG ĐẠT'}")

assert ok_perm and ok_unif, "Kiểm tra rò rỉ KHÔNG ĐẠT theo tiêu chuẩn phân loại mới!"
print("Kiểm tra rò rỉ ĐẠT thành công theo cả hai tiêu chí!")


================ KẾT QUẢ KIỂM TRA RÒ RỈ ================
Cách 1 (Permute - Giữ độ phổ biến): Recall@20 = 0.0396 | Ngưỡng = 0.0828 --> ĐẠT
Cách 2 (Uniform - Phá độ phổ biến): Recall@20 = 0.0085 | Ngưỡng = 0.0165 --> ĐẠT
Kiểm tra rò rỉ ĐẠT thành công theo cả hai tiêu chí!
